# TrustPCB: Similarity-Aware Dataset Split

## Purpose

This notebook constructs and validates a similarity-aware train-validation split for the DsPCBSD+ dataset used in the TrustPCB research project.

The original DsPCBSD+ dataset contains:

- 10,259 images
- 8,208 training images
- 2,051 validation images
- 20,276 bounding-box annotations
- 9 defect classes

A previous dataset audit found:

- no unreadable images,
- no missing image-label pairs,
- no structurally invalid YOLO annotations,
- no exact pixel-identical duplicates, and
- a small number of highly similar image pairs occurring across the supplied training and validation split.


## 1.0: Environment and Dataset Path Verification

Before constructing a new split, the project environment and dataset paths are verified.

This step ensures that:

- the notebook is using the TrustPCB Python environment,
- the original DsPCBSD+ YOLO dataset is accessible,
- the supplied training and validation image folders are available,
- the corresponding label folders are available,
- previously generated audit outputs can be accessed, and
- the directory for new split manifests is identified.

The original files under `data/raw/` will remain unchanged throughout the similarity-aware split construction.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# Project paths

PROJECT_ROOT = Path("/scr/user/danielw9199/TrustPCB")

DATASET_ROOT = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "DsPCBSD_plus"
    / "Data_YOLO"
)

SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

TRAIN_IMAGES = DATASET_ROOT / "images" / "train"
VAL_IMAGES = DATASET_ROOT / "images" / "val"

TRAIN_LABELS = DATASET_ROOT / "labels" / "train"
VAL_LABELS = DATASET_ROOT / "labels" / "val"


# Environment check

print("=== ENVIRONMENT ===")
print("Python executable:")
print(sys.executable)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nDataset root:")
print(DATASET_ROOT)


# Path check

print("\n=== PATH VERIFICATION ===")

paths_to_check = {
    "Training images": TRAIN_IMAGES,
    "Validation images": VAL_IMAGES,
    "Training labels": TRAIN_LABELS,
    "Validation labels": VAL_LABELS,
    "Audit outputs": OUTPUT_DIR,
    "Split directory": SPLIT_DIR,
}

for name, path in paths_to_check.items():
    print(f"{name:20s}: {path.exists()} | {path}")

=== ENVIRONMENT ===
Python executable:
/scr/user/danielw9199/TrustPCB/.conda/bin/python

Project root:
/scr/user/danielw9199/TrustPCB

Dataset root:
/scr/user/danielw9199/TrustPCB/data/raw/DsPCBSD_plus/Data_YOLO

=== PATH VERIFICATION ===
Training images     : True | /scr/user/danielw9199/TrustPCB/data/raw/DsPCBSD_plus/Data_YOLO/images/train
Validation images   : True | /scr/user/danielw9199/TrustPCB/data/raw/DsPCBSD_plus/Data_YOLO/images/val
Training labels     : True | /scr/user/danielw9199/TrustPCB/data/raw/DsPCBSD_plus/Data_YOLO/labels/train
Validation labels   : True | /scr/user/danielw9199/TrustPCB/data/raw/DsPCBSD_plus/Data_YOLO/labels/val
Audit outputs       : True | /scr/user/danielw9199/TrustPCB/outputs/tables
Split directory     : True | /scr/user/danielw9199/TrustPCB/data/splits


## 1.1 Supplied Train-Validation Split Inventory

Before constructing a similarity-aware split, the original DsPCBSD+ train-validation split is reconstructed directly from the official dataset folders.

This establishes the baseline split that will later be compared against the similarity-aware split.

The following checks are performed:

- count training and validation images,
- count corresponding label files,
- verify that image filenames are unique within each split,
- check whether any filename appears in both training and validation, and
- confirm the overall train-validation proportion.

No files are moved or modified during this step.

In [2]:
# 1.1 Reconstruct supplied train-validation split

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

def list_images(folder):
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

def list_labels(folder):
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() == ".txt"
    )

# Load supplied split

train_images = list_images(TRAIN_IMAGES)
val_images = list_images(VAL_IMAGES)

train_labels = list_labels(TRAIN_LABELS)
val_labels = list_labels(VAL_LABELS)

# Basic counts

total_images = len(train_images) + len(val_images)

train_percentage = len(train_images) / total_images * 100
val_percentage = len(val_images) / total_images * 100

print("=== SUPPLIED SPLIT INVENTORY ===")

print("\nTraining:")
print("Images:", len(train_images))
print("Labels:", len(train_labels))

print("\nValidation:")
print("Images:", len(val_images))
print("Labels:", len(val_labels))

print("\nTotal images:", total_images)

print(
    f"Train / validation proportion: "
    f"{train_percentage:.2f}% / {val_percentage:.2f}%"
)

# Check filename uniqueness

train_names = [p.name for p in train_images]
val_names = [p.name for p in val_images]

print("\n=== FILENAME CHECK ===")

print(
    "Unique training filenames:",
    len(set(train_names))
)

print(
    "Unique validation filenames:",
    len(set(val_names))
)

# Check filename overlap across supplied splits

cross_split_name_overlap = sorted(
    set(train_names) & set(val_names)
)

print(
    "Same filename appearing in both splits:",
    len(cross_split_name_overlap)
)

if cross_split_name_overlap:
    print("\nExamples:")
    for name in cross_split_name_overlap[:10]:
        print(" ", name)

# Check image-label pairing

train_image_stems = {p.stem for p in train_images}
train_label_stems = {p.stem for p in train_labels}

val_image_stems = {p.stem for p in val_images}
val_label_stems = {p.stem for p in val_labels}

print("\n=== IMAGE-LABEL PAIRING ===")

print(
    "Train images without labels:",
    len(train_image_stems - train_label_stems)
)

print(
    "Train labels without images:",
    len(train_label_stems - train_image_stems)
)

print(
    "Validation images without labels:",
    len(val_image_stems - val_label_stems)
)

print(
    "Validation labels without images:",
    len(val_label_stems - val_image_stems)
)

=== SUPPLIED SPLIT INVENTORY ===

Training:
Images: 8208
Labels: 8208

Validation:
Images: 2051
Labels: 2051

Total images: 10259
Train / validation proportion: 80.01% / 19.99%

=== FILENAME CHECK ===
Unique training filenames: 8208
Unique validation filenames: 2051
Same filename appearing in both splits: 0

=== IMAGE-LABEL PAIRING ===
Train images without labels: 0
Train labels without images: 0
Validation images without labels: 0
Validation labels without images: 0


## 1.2 Load Confirmed Cross-Split Similarity Pairs

The previous dataset audit identified a small set of high-confidence near-duplicate image pairs that occur across the supplied training and validation split.

This section loads the manually reviewed near-duplicate results so they can be used to guide the construction of a similarity-aware split.

Only the confirmed high-confidence pairs are used at this stage.

In [3]:
# 1.2 Load confirmed cross-split near-duplicate pairs

CONFIRMED_PAIRS_PATH = (
    OUTPUT_DIR
    / "cross_split_high_confidence_near_duplicates.csv"
)

confirmed_pairs = pd.read_csv(CONFIRMED_PAIRS_PATH)

print("=== CONFIRMED CROSS-SPLIT PAIRS ===")
print("File:", CONFIRMED_PAIRS_PATH)
print("Number of confirmed pairs:", len(confirmed_pairs))

print("\nColumns:")
print(list(confirmed_pairs.columns))

print("\nFirst 10 pairs:")
print(
    confirmed_pairs[
        [
            "train_file",
            "val_file",
            "aligned_mae",
            "aligned_pearson"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print("\nUnique affected training images:")
print(confirmed_pairs["train_file"].nunique())

print("\nUnique affected validation images:")
print(confirmed_pairs["val_file"].nunique())

=== CONFIRMED CROSS-SPLIT PAIRS ===
File: /scr/user/danielw9199/TrustPCB/outputs/tables/cross_split_high_confidence_near_duplicates.csv
Number of confirmed pairs: 19

Columns:
['pair_id', 'train_file', 'val_file', 'phash_distance', 'original_mae', 'original_pearson', 'best_dx', 'best_dy', 'aligned_mae', 'aligned_pearson', 'overlap_fraction', 'shift_magnitude', 'mae_improvement', 'manual_label']

First 10 pairs:
    train_file       val_file  aligned_mae  aligned_pearson
  Y_001807.jpg   Y_001811.jpg     0.000058         0.999996
  Y_002229.jpg   Y_002228.jpg     0.000173         0.999988
S_11049285.jpg S_11049386.jpg     0.002039         0.999396
S_11049441.jpg S_11049469.jpg     0.003039         0.998810
S_11903861.jpg S_11903669.jpg     0.004065         0.998657
S_11048293.jpg S_11048199.jpg     0.004169         0.999747
S_11048363.jpg S_11048241.jpg     0.004393         0.999674
   0370652.jpg    0370676.jpg     0.004884         0.999171
S_11917332.jpg S_11917316.jpg     0.005312   

## 1.3 Similarity Group Construction

The confirmed near-duplicate results are currently stored as individual train-validation image pairs.

Before constructing a similarity-aware split, these pairwise relationships are converted into similarity groups.

This is necessary because an image may potentially be related to more than one other image. For example, if image A is similar to image B and image B is similar to image C, all three images should be treated as one connected group rather than as two independent pairs.

The following operations are performed:

- combine the confirmed train-validation pair relationships,
- identify connected groups of related images,
- assign a unique group ID to each similarity group,
- count the number of images in each group, and
- verify how many supplied training and validation images belong to each group.

All images within the same similarity group will later be kept on the same side of the similarity-aware split.

No dataset files are moved or modified during this step.

In [4]:
# 1.3 Construct similarity groups

from collections import defaultdict

# Collect all confirmed pair relationships

pair_edges = list(
    zip(
        confirmed_pairs["train_file"],
        confirmed_pairs["val_file"]
    )
)

# Build an undirected graph

graph = defaultdict(set)

for image_a, image_b in pair_edges:
    graph[image_a].add(image_b)
    graph[image_b].add(image_a)

# Find connected components

visited = set()
similarity_groups = []

for image_name in sorted(graph):

    if image_name in visited:
        continue

    stack = [image_name]
    group = []

    while stack:
        current = stack.pop()

        if current in visited:
            continue

        visited.add(current)
        group.append(current)

        for neighbour in graph[current]:
            if neighbour not in visited:
                stack.append(neighbour)

    similarity_groups.append(sorted(group))

# Map supplied split membership

train_name_set = set(train_names)
val_name_set = set(val_names)

group_records = []

for group_id, group in enumerate(similarity_groups, start=1):

    train_members = [
        name for name in group
        if name in train_name_set
    ]

    val_members = [
        name for name in group
        if name in val_name_set
    ]

    group_records.append({
        "group_id": group_id,
        "group_size": len(group),
        "train_count": len(train_members),
        "val_count": len(val_members),
        "members": ", ".join(group)
    })

similarity_groups_df = pd.DataFrame(group_records)

print("=== SIMILARITY GROUP SUMMARY ===")

print("Confirmed pair relationships:", len(pair_edges))
print("Similarity groups:", len(similarity_groups))
print(
    "Unique images in similarity groups:",
    len(visited)
)

print("\nGroup size distribution:")
print(
    similarity_groups_df["group_size"]
    .value_counts()
    .sort_index()
)

print("\n=== GROUP DETAILS ===")

print(
    similarity_groups_df[
        [
            "group_id",
            "group_size",
            "train_count",
            "val_count",
            "members"
        ]
    ].to_string(index=False)
)

=== SIMILARITY GROUP SUMMARY ===
Confirmed pair relationships: 19
Similarity groups: 19
Unique images in similarity groups: 38

Group size distribution:
group_size
2    19
Name: count, dtype: int64

=== GROUP DETAILS ===
 group_id  group_size  train_count  val_count                        members
        1           2            1          1       0273415.jpg, 0273631.jpg
        2           2            1          1       0370652.jpg, 0370676.jpg
        3           2            1          1     E_000928.jpg, E_000935.jpg
        4           2            1          1 S_10059675.jpg, S_10101113.jpg
        5           2            1          1 S_11048199.jpg, S_11048293.jpg
        6           2            1          1 S_11048241.jpg, S_11048363.jpg
        7           2            1          1 S_11049285.jpg, S_11049386.jpg
        8           2            1          1 S_11049441.jpg, S_11049469.jpg
        9           2            1          1 S_11350344.jpg, S_11350363.jpg
       10

## 1.4 Similarity Group Class Composition

Before assigning the similarity groups to the new train-validation split, the annotation composition of each group is examined.

This is important because moving a similarity group from one side of the split to the other can change the distribution of defect classes. The new split should remove confirmed cross-split similarity while preserving the original class distribution as closely as practical.

The following information is collected for each similarity group:

- the supplied training image,
- the supplied validation image,
- the number of bounding boxes in each image,
- the defect class IDs present in each image, and
- whether both images contain the same set of defect classes.

This information will later be used when deciding how the similarity groups should be assigned in the similarity-aware split.

No train-validation assignments are changed during this step.

In [5]:
# 1.4 Analyse class composition of similarity groups

from collections import Counter

def read_label_classes(label_path):
    classes = []

    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue

        parts = line.split()
        classes.append(int(float(parts[0])))

    return classes

# Build filename-to-label lookup

train_label_lookup = {
    p.stem: p
    for p in train_labels
}

val_label_lookup = {
    p.stem: p
    for p in val_labels
}

# Analyse each similarity group

class_records = []

for group_id, group in enumerate(similarity_groups, start=1):

    train_member = next(
        name for name in group
        if name in train_name_set
    )

    val_member = next(
        name for name in group
        if name in val_name_set
    )

    train_classes = read_label_classes(
        train_label_lookup[Path(train_member).stem]
    )

    val_classes = read_label_classes(
        val_label_lookup[Path(val_member).stem]
    )

    train_class_set = sorted(set(train_classes))
    val_class_set = sorted(set(val_classes))

    class_records.append({
        "group_id": group_id,
        "train_file": train_member,
        "val_file": val_member,
        "train_boxes": len(train_classes),
        "val_boxes": len(val_classes),
        "train_classes": train_class_set,
        "val_classes": val_class_set,
        "same_class_set": train_class_set == val_class_set
    })

group_class_df = pd.DataFrame(class_records)

print("=== SIMILARITY GROUP CLASS COMPOSITION ===")

print(
    group_class_df.to_string(index=False)
)

print("\n=== CLASS-SET AGREEMENT ===")

print(
    group_class_df["same_class_set"]
    .value_counts()
)

# Count all annotations represented by the affected images

affected_train_class_counts = Counter()
affected_val_class_counts = Counter()

for row in group_class_df.itertuples(index=False):

    train_classes = read_label_classes(
        train_label_lookup[Path(row.train_file).stem]
    )

    val_classes = read_label_classes(
        val_label_lookup[Path(row.val_file).stem]
    )

    affected_train_class_counts.update(train_classes)
    affected_val_class_counts.update(val_classes)

print("\n=== AFFECTED TRAINING ANNOTATIONS BY CLASS ===")

for class_id in range(9):
    print(
        f"Class {class_id}: "
        f"{affected_train_class_counts[class_id]}"
    )

print("\n=== AFFECTED VALIDATION ANNOTATIONS BY CLASS ===")

for class_id in range(9):
    print(
        f"Class {class_id}: "
        f"{affected_val_class_counts[class_id]}"
    )

=== SIMILARITY GROUP CLASS COMPOSITION ===
 group_id     train_file       val_file  train_boxes  val_boxes train_classes val_classes  same_class_set
        1    0273415.jpg    0273631.jpg            1          1           [7]         [7]            True
        2    0370652.jpg    0370676.jpg            1          1           [8]         [8]            True
        3   E_000935.jpg   E_000928.jpg            7          7        [0, 1]      [0, 1]            True
        4 S_10059675.jpg S_10101113.jpg            1          1           [1]         [1]            True
        5 S_11048293.jpg S_11048199.jpg            1          1           [1]         [1]            True
        6 S_11048363.jpg S_11048241.jpg            1          1           [4]         [4]            True
        7 S_11049285.jpg S_11049386.jpg            1          1           [4]         [4]            True
        8 S_11049441.jpg S_11049469.jpg            1          1           [4]         [4]            True
   

## 1.5 Supplied Split Class Distribution

The class distribution of the supplied train-validation split is calculated before creating the similarity-aware split.

This gives us a reference to compare against later.

The following are checked:

- annotation count for each class,
- class percentage in training and validation, and
- difference between the two splits.

The similarity-aware split should keep these distributions as close as possible to the supplied split.

In [6]:
# 1.5 Calculate supplied split class distribution

def count_classes(label_files):
    counts = Counter()

    for label_path in label_files:
        classes = read_label_classes(label_path)
        counts.update(classes)

    return counts

train_class_counts = count_classes(train_labels)
val_class_counts = count_classes(val_labels)

train_total_boxes = sum(train_class_counts.values())
val_total_boxes = sum(val_class_counts.values())

class_distribution_records = []

for class_id in range(9):

    train_count = train_class_counts[class_id]
    val_count = val_class_counts[class_id]

    train_pct = train_count / train_total_boxes * 100
    val_pct = val_count / val_total_boxes * 100

    class_distribution_records.append({
        "class_id": class_id,
        "train_count": train_count,
        "train_pct": train_pct,
        "val_count": val_count,
        "val_pct": val_pct,
        "pct_difference": abs(train_pct - val_pct)
    })

supplied_class_df = pd.DataFrame(class_distribution_records)

print("=== SUPPLIED SPLIT CLASS DISTRIBUTION ===")

print(
    supplied_class_df.round({
        "train_pct": 2,
        "val_pct": 2,
        "pct_difference": 2
    }).to_string(index=False)
)

print("\nTraining annotations:", train_total_boxes)
print("Validation annotations:", val_total_boxes)

print(
    "\nLargest class percentage difference:",
    f"{supplied_class_df['pct_difference'].max():.2f}%"
)

=== SUPPLIED SPLIT CLASS DISTRIBUTION ===
 class_id  train_count  train_pct  val_count  val_pct  pct_difference
        0          746       4.61        169     4.13            0.48
        1         3655      22.58        929    22.70            0.12
        2         1308       8.08        285     6.96            1.12
        3         1432       8.85        338     8.26            0.59
        4         1983      12.25        546    13.34            1.09
        5         2275      14.06        608    14.86            0.80
        6         2042      12.62        448    10.95            1.67
        7         1409       8.71        423    10.34            1.63
        8         1334       8.24        346     8.46            0.21

Training annotations: 16184
Validation annotations: 4092

Largest class percentage difference: 1.67%


## 1.6 Similarity-Aware Split Strategy

The similarity-aware split will be created by modifying the supplied split as little as possible.

The 19 confirmed similarity groups each contain:

- 1 training image,
- 1 validation image, and
- 2 images in total.

Both images from each similarity group must be assigned to the same split.

Instead of creating a completely new random split, the supplied split is kept unchanged for all unaffected images.

For the 19 similarity groups:

- some groups will be assigned entirely to training,
- the remaining groups will be assigned entirely to validation,
- only one image from each group needs to change split, and
- the final train-validation ratio should remain as close as possible to the original 80/20 ratio.

This minimal-change approach allows the supplied and similarity-aware splits to remain directly comparable.

In [7]:
# 1.6 Check feasible similarity-group allocation

n_groups = len(similarity_groups)

original_train = len(train_images)
original_val = len(val_images)

affected_train = 19
affected_val = 19

unaffected_train = original_train - affected_train
unaffected_val = original_val - affected_val

print("=== SIMILARITY-AWARE SPLIT STRATEGY ===")

print("Similarity groups:", n_groups)
print("Images per group: 2")
print("Images that must change split:", n_groups)

print("\nOriginal split:")
print("Training:", original_train)
print("Validation:", original_val)

print("\nUnaffected images:")
print("Training:", unaffected_train)
print("Validation:", unaffected_val)

print("\n=== CLOSEST POSSIBLE 80/20 ALLOCATIONS ===")

for groups_to_train in [9, 10]:

    groups_to_val = n_groups - groups_to_train

    new_train = unaffected_train + (groups_to_train * 2)
    new_val = unaffected_val + (groups_to_val * 2)

    total = new_train + new_val

    train_pct = new_train / total * 100
    val_pct = new_val / total * 100

    print(
        f"\n{groups_to_train} groups -> train, "
        f"{groups_to_val} groups -> validation"
    )

    print(
        f"Training: {new_train} ({train_pct:.2f}%)"
    )

    print(
        f"Validation: {new_val} ({val_pct:.2f}%)"
    )

=== SIMILARITY-AWARE SPLIT STRATEGY ===
Similarity groups: 19
Images per group: 2
Images that must change split: 19

Original split:
Training: 8208
Validation: 2051

Unaffected images:
Training: 8189
Validation: 2032

=== CLOSEST POSSIBLE 80/20 ALLOCATIONS ===

9 groups -> train, 10 groups -> validation
Training: 8207 (80.00%)
Validation: 2052 (20.00%)

10 groups -> train, 9 groups -> validation
Training: 8209 (80.02%)
Validation: 2050 (19.98%)


## 1.7 Select Similarity Group Allocation

The 19 similarity groups must be assigned entirely to either training or validation.

Two group-count allocations can keep the dataset close to the original 80/20 split:

- 9 groups to training and 10 groups to validation, or
- 10 groups to training and 9 groups to validation.

The specific groups should not be selected randomly.

Each possible allocation is compared based on:

- training and validation class distribution,
- difference from the supplied split class percentages, and
- final train-validation image count.

The allocation with the smallest class-distribution change will be selected.

In [8]:
# 1.7 Select similarity group allocation

from itertools import combinations

# Get annotation counts for each similarity group

group_annotation_counts = {}

for row in group_class_df.itertuples(index=False):

    train_classes = read_label_classes(
        train_label_lookup[Path(row.train_file).stem]
    )

    val_classes = read_label_classes(
        val_label_lookup[Path(row.val_file).stem]
    )

    counts = Counter(train_classes + val_classes)

    group_annotation_counts[row.group_id] = np.array(
        [counts[class_id] for class_id in range(9)],
        dtype=int
    )

# Get annotation counts from unaffected images

affected_train_names = set(confirmed_pairs["train_file"])
affected_val_names = set(confirmed_pairs["val_file"])

unaffected_train_labels = [
    p for p in train_labels
    if f"{p.stem}.jpg" not in affected_train_names
]

unaffected_val_labels = [
    p for p in val_labels
    if f"{p.stem}.jpg" not in affected_val_names
]

unaffected_train_counts = count_classes(unaffected_train_labels)
unaffected_val_counts = count_classes(unaffected_val_labels)

base_train_counts = np.array(
    [unaffected_train_counts[i] for i in range(9)]
)

base_val_counts = np.array(
    [unaffected_val_counts[i] for i in range(9)]
)

target_train_pct = supplied_class_df["train_pct"].to_numpy()
target_val_pct = supplied_class_df["val_pct"].to_numpy()

group_ids = sorted(group_annotation_counts.keys())

results = []

# Check both near-80/20 group allocations

for n_train_groups in [9, 10]:

    for train_group_tuple in combinations(
        group_ids,
        n_train_groups
    ):

        train_group_set = set(train_group_tuple)

        train_counts_candidate = base_train_counts.copy()
        val_counts_candidate = base_val_counts.copy()

        for group_id in group_ids:

            if group_id in train_group_set:
                train_counts_candidate += group_annotation_counts[group_id]
            else:
                val_counts_candidate += group_annotation_counts[group_id]

        train_pct_candidate = (
            train_counts_candidate
            / train_counts_candidate.sum()
            * 100
        )

        val_pct_candidate = (
            val_counts_candidate
            / val_counts_candidate.sum()
            * 100
        )

        train_diff = np.abs(
            train_pct_candidate - target_train_pct
        )

        val_diff = np.abs(
            val_pct_candidate - target_val_pct
        )

        mean_difference = np.mean(
            np.concatenate([train_diff, val_diff])
        )

        max_difference = np.max(
            np.concatenate([train_diff, val_diff])
        )

        results.append({
            "train_groups": tuple(sorted(train_group_set)),
            "val_groups": tuple(
                sorted(set(group_ids) - train_group_set)
            ),
            "n_train_groups": n_train_groups,
            "mean_pct_difference": mean_difference,
            "max_pct_difference": max_difference,
            "train_counts": train_counts_candidate,
            "val_counts": val_counts_candidate
        })

allocation_results = sorted(
    results,
    key=lambda x: (
        x["mean_pct_difference"],
        x["max_pct_difference"]
    )
)

best_allocation = allocation_results[0]

print("=== BEST SIMILARITY GROUP ALLOCATION ===")

print(
    "Groups assigned to training:",
    best_allocation["train_groups"]
)

print(
    "Groups assigned to validation:",
    best_allocation["val_groups"]
)

print(
    "\nNumber of training groups:",
    len(best_allocation["train_groups"])
)

print(
    "Number of validation groups:",
    len(best_allocation["val_groups"])
)

print(
    "\nMean class percentage change:",
    f"{best_allocation['mean_pct_difference']:.4f}%"
)

print(
    "Largest class percentage change:",
    f"{best_allocation['max_pct_difference']:.4f}%"
)

# Calculate final image counts

new_train_count = (
    unaffected_train
    + 2 * len(best_allocation["train_groups"])
)

new_val_count = (
    unaffected_val
    + 2 * len(best_allocation["val_groups"])
)

print("\nFinal image counts:")
print(
    f"Training: {new_train_count} "
    f"({new_train_count / total_images * 100:.2f}%)"
)

print(
    f"Validation: {new_val_count} "
    f"({new_val_count / total_images * 100:.2f}%)"
)

=== BEST SIMILARITY GROUP ALLOCATION ===
Groups assigned to training: (1, 2, 4, 5, 6, 9, 10, 14, 16)
Groups assigned to validation: (3, 7, 8, 11, 12, 13, 15, 17, 18, 19)

Number of training groups: 9
Number of validation groups: 10

Mean class percentage change: 0.0062%
Largest class percentage change: 0.0177%

Final image counts:
Training: 8207 (80.00%)
Validation: 2052 (20.00%)


## 1.8 Verify Selected Split Distribution

The selected group allocation is checked before creating the final similarity-aware split.

The following are compared:

- supplied training class distribution,
- similarity-aware training class distribution,
- supplied validation class distribution,
- similarity-aware validation class distribution, and
- percentage-point change for each class.

The purpose is to confirm that removing cross-split similarity does not significantly change the class composition of the dataset.

In [9]:
# 1.8 Verify selected similarity-aware class distribution

new_train_counts = best_allocation["train_counts"]
new_val_counts = best_allocation["val_counts"]

new_train_pct = (
    new_train_counts
    / new_train_counts.sum()
    * 100
)

new_val_pct = (
    new_val_counts
    / new_val_counts.sum()
    * 100
)

comparison_records = []

for class_id in range(9):

    supplied_train_pct = supplied_class_df.loc[
        supplied_class_df["class_id"] == class_id,
        "train_pct"
    ].iloc[0]

    supplied_val_pct = supplied_class_df.loc[
        supplied_class_df["class_id"] == class_id,
        "val_pct"
    ].iloc[0]

    comparison_records.append({
        "class_id": class_id,
        "supplied_train_pct": supplied_train_pct,
        "new_train_pct": new_train_pct[class_id],
        "train_change": (
            new_train_pct[class_id]
            - supplied_train_pct
        ),
        "supplied_val_pct": supplied_val_pct,
        "new_val_pct": new_val_pct[class_id],
        "val_change": (
            new_val_pct[class_id]
            - supplied_val_pct
        )
    })

split_comparison_df = pd.DataFrame(
    comparison_records
)

print("=== CLASS DISTRIBUTION COMPARISON ===")

print(
    split_comparison_df.round(3).to_string(index=False)
)

print("\n=== MAXIMUM ABSOLUTE CHANGE ===")

print(
    "Training:",
    f"{split_comparison_df['train_change'].abs().max():.4f} percentage points"
)

print(
    "Validation:",
    f"{split_comparison_df['val_change'].abs().max():.4f} percentage points"
)

print("\n=== FINAL SPLIT SIZE ===")

print(
    f"Training: {new_train_count} "
    f"({new_train_count / total_images * 100:.2f}%)"
)

print(
    f"Validation: {new_val_count} "
    f"({new_val_count / total_images * 100:.2f}%)"
)

=== CLASS DISTRIBUTION COMPARISON ===
 class_id  supplied_train_pct  new_train_pct  train_change  supplied_val_pct  new_val_pct  val_change
        0               4.609          4.605        -0.004             4.130        4.147       0.017
        1              22.584         22.581        -0.003            22.703       22.713       0.010
        2               8.082          8.086         0.003             6.965        6.953      -0.012
        3               8.848          8.852         0.004             8.260        8.246      -0.014
        4              12.253         12.252        -0.001            13.343       13.345       0.002
        5              14.057         14.057        -0.000            14.858       14.857      -0.001
        6              12.617         12.617        -0.001            10.948       10.954       0.006
        7               8.706          8.710         0.004            10.337       10.320      -0.018
        8               8.243          8.240

## 1.9 Create Similarity-Aware Split Manifests

The selected group allocation is now used to create the similarity-aware train-validation split.

The original dataset files are not moved or modified.

Instead, split manifests are created to record:

- which images belong to training,
- which images belong to validation,
- the original supplied split,
- the new similarity-aware split, and
- which images changed split.

All images within the same confirmed similarity group are assigned to the same split.

In [10]:
# 1.9 Create similarity-aware split manifests

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# Map every image filename to its original location

image_path_lookup = {}

for path in train_images:
    image_path_lookup[path.name] = path

for path in val_images:
    image_path_lookup[path.name] = path

# Start from the supplied split

new_train_names = set(train_names)
new_val_names = set(val_names)

train_group_ids = set(best_allocation["train_groups"])
val_group_ids = set(best_allocation["val_groups"])

# Assign every similarity group entirely to one split

for group_id, group in enumerate(similarity_groups, start=1):

    for image_name in group:
        new_train_names.discard(image_name)
        new_val_names.discard(image_name)

    if group_id in train_group_ids:
        new_train_names.update(group)

    elif group_id in val_group_ids:
        new_val_names.update(group)

    else:
        raise RuntimeError(
            f"Group {group_id} was not assigned."
        )

# Sort manifests

new_train_names = sorted(new_train_names)
new_val_names = sorted(new_val_names)

# Identify images that changed split

moved_train_to_val = sorted(
    set(train_names) & set(new_val_names)
)

moved_val_to_train = sorted(
    set(val_names) & set(new_train_names)
)

print("=== SIMILARITY-AWARE SPLIT ===")

print("Training images:", len(new_train_names))
print("Validation images:", len(new_val_names))

print("\nImages moved train -> validation:")
print(len(moved_train_to_val))

print("\nImages moved validation -> train:")
print(len(moved_val_to_train))

print("\nTotal images moved:")
print(
    len(moved_train_to_val)
    + len(moved_val_to_train)
)

# Save filename manifests

(SPLIT_DIR / "supplied_train.txt").write_text(
    "\n".join(sorted(train_names)) + "\n"
)

(SPLIT_DIR / "supplied_val.txt").write_text(
    "\n".join(sorted(val_names)) + "\n"
)

(SPLIT_DIR / "similarity_aware_train.txt").write_text(
    "\n".join(new_train_names) + "\n"
)

(SPLIT_DIR / "similarity_aware_val.txt").write_text(
    "\n".join(new_val_names) + "\n"
)

# Save complete split mapping

manifest_records = []

for image_name in sorted(set(train_names) | set(val_names)):

    original_split = (
        "train"
        if image_name in set(train_names)
        else "val"
    )

    similarity_split = (
        "train"
        if image_name in set(new_train_names)
        else "val"
    )

    manifest_records.append({
        "image": image_name,
        "supplied_split": original_split,
        "similarity_aware_split": similarity_split,
        "changed_split": original_split != similarity_split
    })

split_manifest_df = pd.DataFrame(manifest_records)

split_manifest_path = (
    SPLIT_DIR
    / "similarity_aware_split_manifest.csv"
)

split_manifest_df.to_csv(
    split_manifest_path,
    index=False
)

print("\n=== SAVED FILES ===")

for path in [
    SPLIT_DIR / "supplied_train.txt",
    SPLIT_DIR / "supplied_val.txt",
    SPLIT_DIR / "similarity_aware_train.txt",
    SPLIT_DIR / "similarity_aware_val.txt",
    split_manifest_path
]:
    print(path)

=== SIMILARITY-AWARE SPLIT ===
Training images: 8207
Validation images: 2052

Images moved train -> validation:
10

Images moved validation -> train:
9

Total images moved:
19

=== SAVED FILES ===
/scr/user/danielw9199/TrustPCB/data/splits/supplied_train.txt
/scr/user/danielw9199/TrustPCB/data/splits/supplied_val.txt
/scr/user/danielw9199/TrustPCB/data/splits/similarity_aware_train.txt
/scr/user/danielw9199/TrustPCB/data/splits/similarity_aware_val.txt
/scr/user/danielw9199/TrustPCB/data/splits/similarity_aware_split_manifest.csv


## 1.10 Validate Similarity-Aware Split

The new similarity-aware split is checked before it is used for training.

The following are verified:

- no image appears in both training and validation,
- all 10,259 images are included,
- all confirmed similarity groups stay within one split,
- final train-validation size remains close to 80/20, and
- the number of moved images matches the expected 19 images.

This confirms that the new split removes the confirmed cross-split similarity without changing the dataset unnecessarily.

In [11]:
# 1.10 Validate similarity-aware split

new_train_set = set(new_train_names)
new_val_set = set(new_val_names)

all_original_images = set(train_names) | set(val_names)
all_new_images = new_train_set | new_val_set

# Check train-validation overlap

split_overlap = new_train_set & new_val_set

# Check missing or unexpected images

missing_images = all_original_images - all_new_images
unexpected_images = all_new_images - all_original_images

# Check confirmed similarity groups

cross_split_groups = []

for group_id, group in enumerate(similarity_groups, start=1):

    group_train = [
        name for name in group
        if name in new_train_set
    ]

    group_val = [
        name for name in group
        if name in new_val_set
    ]

    if group_train and group_val:
        cross_split_groups.append({
            "group_id": group_id,
            "train_members": group_train,
            "val_members": group_val
        })

# Count changed assignments

changed_images = split_manifest_df[
    split_manifest_df["changed_split"]
]

print("=== FINAL SPLIT VALIDATION ===")

print("Training images:", len(new_train_set))
print("Validation images:", len(new_val_set))
print("Total images:", len(all_new_images))

print(
    "\nTrain / validation proportion:",
    f"{len(new_train_set) / len(all_new_images) * 100:.2f}% / "
    f"{len(new_val_set) / len(all_new_images) * 100:.2f}%"
)

print("\nImages appearing in both splits:", len(split_overlap))
print("Missing images:", len(missing_images))
print("Unexpected images:", len(unexpected_images))

print(
    "Confirmed similarity groups crossing splits:",
    len(cross_split_groups)
)

print(
    "Images with changed split assignment:",
    len(changed_images)
)

if cross_split_groups:
    print("\nCross-split similarity groups:")
    for group in cross_split_groups:
        print(group)

# Final pass/fail

validation_passed = (
    len(new_train_set) + len(new_val_set) == 10259
    and len(split_overlap) == 0
    and len(missing_images) == 0
    and len(unexpected_images) == 0
    and len(cross_split_groups) == 0
    and len(changed_images) == 19
)

print("\n=== VALIDATION RESULT ===")

if validation_passed:
    print("PASS")
else:
    print("FAIL")

=== FINAL SPLIT VALIDATION ===
Training images: 8207
Validation images: 2052
Total images: 10259

Train / validation proportion: 80.00% / 20.00%

Images appearing in both splits: 0
Missing images: 0
Unexpected images: 0
Confirmed similarity groups crossing splits: 0
Images with changed split assignment: 19

=== VALIDATION RESULT ===
PASS


## 1.11 Summary

A similarity-aware train-validation split was constructed from the supplied DsPCBSD+ split.

The main results are:

- original split: 8,208 training and 2,051 validation images,
- new split: 8,207 training and 2,052 validation images,
- final proportion: 80.00% training and 20.00% validation,
- 19 confirmed cross-split near-duplicate groups were considered,
- 19 images changed split assignment,
- no image appears in both splits,
- no images are missing or added,
- no confirmed similarity group remains across training and validation, and
- maximum class-distribution change is only 0.0177 percentage points.

The similarity-aware split therefore removes the confirmed high-confidence cross-split similarities while keeping the dataset size and class distribution almost unchanged.

The split manifests are saved under `data/splits/` for use in later experiments.